### LARGE SCALE BILLION DOLLAR COMPANIES Sustainability REPORTS RAG Pipeline
This notebook implements an advanced, production-grade Retrieval-Augmented Generation (RAG) system to analyze corporate sustainability reports. 

### Core Architecture & Frameworks Used:
1. **Data Parsing:** `Docling` using the lightweight `pypdfium2` backend to bypass native system memory bugs and extract structure-aware content blocks.
2. **Text Chunking:** Docling's `HybridChunker` to map true layout boundaries and inject parent header lineages into text slices.
3. **Vector Database:** `FAISS (Facebook AI Similarity Search)` for lightning-fast, dense in-memory structural token matrices.
4. **Embedding Model:** `BAAI/bge-base-en-v1.5` (Top-tier open-source embeddings natively generating 768-dimensional normalized vectors).
5. **Neural Reranker:** `BAAI/bge-reranker-base` (Cross-Encoder framework that scores query-context pairs simultaneously to collapse raw search pools down to highly relevant points).
6. **LLM Orchestration:** `LangChain` paired with `openai/gpt-oss-120b` to deliver factually grounded responses.


In [3]:
import os
import time
import json
from pathlib import Path
from typing import List, Dict, Any, Tuple
from getpass import getpass

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Docling Infrastructure
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

# LlamaIndex Structural Ecosystem
from llama_index.core import VectorStoreIndex, Settings, SimpleDirectoryReader, PromptTemplate, StorageContext, load_index_from_storage
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.llms.groq import Groq
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.vector_stores.types import MetadataFilter, MetadataFilters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LlamaIndex Core Evaluators
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

print("Modular RAG environment dependencies configured completely!")


d:\G_AI\logicmojo-data-science-ai-nov-2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modular RAG environment dependencies configured completely!



### Convert PDF to Markdown

Markdown is widely considered the best intermediary format for a RAG pipeline. It preserves structural hierarchy natively using tags (like # Heading 1, | Table |, and **bold**), allowing downstream semantic chunkers to understand exactly where a section starts and ends.

**Docling (IBM Research)** : The best default choice for RAG pipelines. It is MIT-licensed, completely free, and runs efficiently on CPU. It features TableFormer, a specialized deep-learning model designed specifically to extract complex rows, columns, and nested table headers perfectly into Markdown.

### **Step 1** : Loading all the pdfs and extracting the pdfs contents into each md files.

In [4]:
from pathlib import Path
import time

def process_pdf_rag_pipeline(input_dir: str, output_dir: str):
    """
    Reads PDFs from input_dir, converts them to individual Markdown files using Docling,
    saves them to output_dir, and prepares them for semantic chunking.
    
    MODIFICATION: Automatically skips files that already have a matching .md file in the output directory.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("Initializing Docling Document Converter (loading models)...")
    
    # Configure limits on the pipeline options object
    pipeline_options = PdfPipelineOptions()
    pipeline_options.layout_batch_size = 1        # Forces processing of 1 page at a time
    pipeline_options.do_ocr = False 
    
    # --- FIXED BACKEND REGISTRATION ---
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, 
                backend=PyPdfiumDocumentBackend  # <-- Explicitly sets the stable backend
            )
        }
    )   
    
    # Find all PDFs in the source folder
    pdf_files = list(input_path.glob("*.pdf"))
    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return
        
    print(f"Found {len(pdf_files)} PDF files in total.")
    
    # ----------------------------------------------------
    # FIX: Filter out PDFs that have already been converted
    # ----------------------------------------------------
    pending_pdf_files = []
    for pdf_file in pdf_files:
        md_filename = pdf_file.stem + ".md"
        md_file_path = output_path / md_filename
        
        if md_file_path.exists():
            print(f" Skipping: {pdf_file.name} (Already converted to {md_filename})")
        else:
            pending_pdf_files.append(pdf_file)
            
    if not pending_pdf_files:
        print("🎉 All PDF files have already been converted to Markdown!")
        return
        
    print(f"Processing remaining {len(pending_pdf_files)} pending PDF file(s)...\n")
    
    # 2. Iterate and convert each pending PDF
    for pdf_file in pending_pdf_files:
        start_time = time.time()
        print(f" Processing: {pdf_file.name}")
        
        try:
            # Convert PDF using layout-aware models
            result = converter.convert(pdf_file)
            
            # Export the layout structure directly to clean Markdown text
            markdown_content = result.document.export_to_markdown()
            
            # Define output path (changing extension from .pdf to .md)
            md_filename = pdf_file.stem + ".md"
            md_file_path = output_path / md_filename
            
            # Save the clean Markdown file locally
            with open(md_file_path, "w", encoding="utf-8") as f:
                f.write(markdown_content)
                
            elapsed_time = time.time() - start_time
            print(f" Saved to: {md_file_path} ({elapsed_time:.2f}s)")
            
            # 3. Step Into Chunking Strategy
            print(f"--- Chunking Content for: {pdf_file.name} ---")
            
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1024,
                chunk_overlap=200,
                separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", " ", ""]
            )
            
            # Generate the final chunk split
            chunks = text_splitter.split_text(markdown_content)
            print(f"Generated {len(chunks)} text chunks ready for embedding.\n")
            
            # Example metadata dictionary tracking structural provenance 
            for idx, chunk in enumerate(chunks[:2]): # Print first two chunks as a quick preview
                metadata = {
                    "source_file": pdf_file.name,
                    "chunk_index": idx
                }
                print(f"Chunk {idx} Metadata: {metadata}")
                
        except Exception as e:
            print(f" Failed to process {pdf_file.name}. Error: {str(e)}\n")

In [15]:
# Define your local data pathways
SOURCE_DATA_FOLDER = "./data"          # Place all your 10+ PDFs here
OUTPUT_MARKDOWN_FOLDER = "./output_md" # Where all your converted Markdown files go

# Run the ingestion layer
process_pdf_rag_pipeline(SOURCE_DATA_FOLDER, OUTPUT_MARKDOWN_FOLDER)

Initializing Docling Document Converter (loading models)...
Found 11 PDF files in total.
 Skipping: 2025_Climate-Related_Disclosures_Report_General_Motors.pdf (Already converted to 2025_Climate-Related_Disclosures_Report_General_Motors.md)
 Skipping: 2026-Microsoft-Environmental-Sustainability-Report-PDF.pdf (Already converted to 2026-Microsoft-Environmental-Sustainability-Report-PDF.md)
 Skipping: Apple_Environmental_Progress_Report_2024.pdf (Already converted to Apple_Environmental_Progress_Report_2024.md)
 Skipping: google-2024-environmental-report.pdf (Already converted to google-2024-environmental-report.md)
 Skipping: google-sustainable-2026-download-1699a2e1.pdf (Already converted to google-sustainable-2026-download-1699a2e1.md)
 Skipping: ikea-cooking-eating-report-2026.pdf (Already converted to ikea-cooking-eating-report-2026.md)
 Skipping: Responsible-AI-Transparency-Report-2026.pdf (Already converted to Responsible-AI-Transparency-Report-2026.md)
 Skipping: The-Future-of-Hos

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 4336.02it/s]


 Saved to: output_md\2024-coco-cola-environmental-update.md (36.11s)
--- Chunking Content for: 2024-coco-cola-environmental-update.pdf ---
Generated 90 text chunks ready for embedding.

Chunk 0 Metadata: {'source_file': '2024-coco-cola-environmental-update.pdf', 'chunk_index': 0}
Chunk 1 Metadata: {'source_file': '2024-coco-cola-environmental-update.pdf', 'chunk_index': 1}
 Processing: sugar-study-methodology-overview.pdf
 Saved to: output_md\sugar-study-methodology-overview.md (15.34s)
--- Chunking Content for: sugar-study-methodology-overview.pdf ---
Generated 60 text chunks ready for embedding.

Chunk 0 Metadata: {'source_file': 'sugar-study-methodology-overview.pdf', 'chunk_index': 0}
Chunk 1 Metadata: {'source_file': 'sugar-study-methodology-overview.pdf', 'chunk_index': 1}


### Step 2 : Reading .md files Chunk them Embedding & Store in vector index

In [5]:
def build_persistent_vector_index(
    markdown_dir: str, 
    groq_api_key: str,
    embedding_model_name: str = "BAAI/bge-base-en-v1.5",
    persist_dir: str = "./storage_index"
) -> VectorStoreIndex:
    """
    Initializes global model contexts using active Groq production model IDs, 
    checks disk cache for pre-computed vectors, and loads or creates indices.
    """
    # Using stable Groq model endpoint targets
    TARGET_GROQ_MODEL = "openai/gpt-oss-120b" 
    
    print(f" Overriding global runtime settings to active model: '{TARGET_GROQ_MODEL}'")
    Settings.llm = Groq(model=TARGET_GROQ_MODEL, api_key=groq_api_key)
    Settings.embed_model = HuggingFaceEmbedding(model_name=embedding_model_name)
    
    persist_path = Path(persist_dir)
    
    # 2. HARD DRIVE CACHE CHECK: If index files exist on disk, restore them instantly
    if persist_path.exists() and (persist_path / "docstore.json").exists():
        print(f" Cache Hit: Restoring persistent vector index tracks out of disk storage from '{persist_dir}'...")
        start_time = time.time()
        
        storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
        sustainability_index = load_index_from_storage(storage_context)
        
        print(f" Persistent index loaded directly from drive in {time.time() - start_time:.2f}s!")
        return sustainability_index

    # 3. CACHE MISS: Read raw text files and generate fresh embeddings
    print(" Cache Miss: Permanent storage index not found. Compiling index layers from scratch...")
    markdown_path = Path(markdown_dir)
    md_files = list(markdown_path.glob("*.md"))
    
    if not md_files:
        raise FileNotFoundError(f"No generated markdown logs discovered in folder: '{markdown_dir}'")
        
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024, chunk_overlap=200,
        separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", " ", ""]
    )
    
    llama_nodes = []
    print(f" Processing text splitting for {len(md_files)} markdown records...")
    
    for md_file in md_files:
        raw_name = md_file.stem.lower()
        # Create a simple clean identifier string tag for metadata filtering
        company_tag = raw_name.replace("2024", "").replace("2025", "").replace("2026", "").replace("_", " ").replace("-", " ").strip().split()[0]
        
        try:
            with open(md_file, "r", encoding="utf-8") as f:
                content = f.read()
            chunks = text_splitter.split_text(content)
            
            for idx, chunk in enumerate(chunks):
                node = TextNode(
                    text=chunk,
                    metadata={"source_file": md_file.name, "chunk_index": idx, "company": company_tag}
                )
                llama_nodes.append(node)
        except Exception as e:
            print(f"   Error processing file {md_file.name}: {str(e)}")
            
    print(f" Generating vector embeddings over {len(llama_nodes)} text segment blocks...")
    start_time = time.time()
    
    # -----------------------------------------------------------------
    # CRITICAL FIX 1: Explicitly pass nodes using the `nodes=` keyword argument
    # -----------------------------------------------------------------
    sustainability_index = VectorStoreIndex(nodes=llama_nodes, show_progress=True)
    print(f" Vectors generated in {time.time() - start_time:.2f}s")
    
    # -----------------------------------------------------------------
    # CRITICAL FIX 2: Ensure path creation and absolute string serialization
    # -----------------------------------------------------------------
    print(f" Committing vector matrix tables permanently to hard drive path: '{persist_dir}'")
    persist_path.mkdir(parents=True, exist_ok=True)
    
    # Target storage layer context directory directly
    sustainability_index.storage_context.persist(persist_dir=str(persist_path))
    print(" Storage save completed successfully!")
    
    return sustainability_index


In [8]:
# Generate your secure Groq key from ://groq.com
MY_GROQ_API_KEY = getpass("Enter your GROQ API KEY: ")

In [6]:
# Using the exact directory variable names defined in your folder environment structure
OUTPUT_MARKDOWN_FOLDER = "./output_md"  # The folder containing your 9+ .md files
BGE_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# Generate your secure Groq key from ://groq.com
MY_GROQ_API_KEY = getpass("Enter your GROQ API KEY: ")

# Execute the persistent database instantiation layer
# This variable name 'sustainability_vector_index' now points to a secure disk-backed asset!
sustainability_vector_index = build_persistent_vector_index(
    markdown_dir=OUTPUT_MARKDOWN_FOLDER,
    embedding_model_name=BGE_EMBEDDING_MODEL,
    groq_api_key=MY_GROQ_API_KEY,
    persist_dir="./storage_index"  # Folder where JSON files will be written
)

print("\n Next steps: The index is generated and ready to connect to retrieval and evaluation chains!")

 Overriding global runtime settings to active model: 'openai/gpt-oss-120b'


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 568.15it/s]


 Cache Hit: Restoring persistent vector index tracks out of disk storage from './storage_index'...
 Persistent index loaded directly from drive in 17.55s!

 Next steps: The index is generated and ready to connect to retrieval and evaluation chains!


In LlamaIndex, the query_engine is a pre-assembled pipeline. When you call .query(query), it automatically triggers a sequential, multi-step process.
```
[query] (Raw String Text)
    |
    v
 1. Settings.embed_model.get_query_embedding(query) 
    |---> Converts text to a 768-dimensional float32 vector array
    |
    v
 2. index.as_retriever(similarity_top_k=5)
    |---> Computes vector dot-products against the text chunks in memory
    |---> Grabs the Top 5 closest math coordinates
    |
    v
 3. reranker_processor.postprocess_nodes(nodes, query_str=query)
    |---> Feeds raw query text + 5 chunks into the Cross-Encoder neural network
    |---> Rearranges and slices down to the absolute top 3 nodes
    |
    v
 4. text_qa_template.format(query_str=query, context_str=top_3_chunks)
    |---> Places your raw text into the {query_str} slot of your markdown prompt
    |
    v
 [Groq LLM Inference Completion Model]
 ```

### Step 3 : Retrival Pipeline using LLama_index Query Engine

In [7]:
from pathlib import Path
import time
from typing import Any
from llama_index.core import StorageContext, load_index_from_storage, PromptTemplate
from llama_index.core.postprocessor import SentenceTransformerRerank

def query_persistent_sustainability_rag(
    query: str, 
    persist_dir: str = "./storage_index", 
    reranker_model_name: str = "BAAI/bge-reranker-base"
) -> Any:
    """
    Loads the persistent index directly from disk, retrieves top coarse vector blocks, 
    neural reranks them, and answers via Groq using a strict grounded prompt.
    """
    persist_path = Path(persist_dir)
    
    if not persist_path.exists() or not (persist_path / "docstore.json").exists():
        raise FileNotFoundError(
            f"No persistent index found at '{persist_dir}'. "
            "Please run your index building function first!"
        )
    
    # 1. Load storage context and index straight from disk
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
    index = load_index_from_storage(storage_context)
    
    # 2. Initialize the cross-encoder reranking processor
    reranker_processor = SentenceTransformerRerank(model=reranker_model_name, top_n=3)
    
    # 3. Compile query runtime engine targeting top coarse matches and reranking
    query_engine = index.as_query_engine(
        similarity_top_k=15,
        node_postprocessors=[reranker_processor]
    )
    
    # 4. Rigorous prompt template to block hallucinations and force inline document citations
    grounded_qa_template = (
        "You are an elite corporate governance and environmental data audit assistant.\n"
        "Your task is to synthesize a structured answer using ONLY the factual context blocks provided below.\n\n"
        "CRITICAL SYSTEM OPERATIONAL RULES:\n"
        "1. If the provided context blocks do not contain clear factual evidence to answer the question, "
        "state explicitly: 'Information not found in context.'\n"
        "2. Do not extrapolate, assume, or pull outside industry knowledge. Rely entirely on the text facts.\n"
        "3. Strictly ground every single claim back to its respective original source file using explicit "
        "inline citations, for example: (Source File: name_of_document.md).\n"
        "4. Keep your output highly concise, direct, objective, and structured as clean Markdown prose.\n\n"
        "FACTUAL DATA CONTEXT LOGS:\n{context_str}\n\n"
        "USER QUESTION: {query_str}\n"
        "GROUNDED AUDIT ANSWER:"
    )
    
    text_qa_template = PromptTemplate(grounded_qa_template)
    query_engine.update_prompts({"response_synthesizer:text_qa_template": text_qa_template})
    
    # 5. Query conversion and execution over persistent matrix layers
    start_time = time.time()
    response = query_engine.query(query)
    print(f"Search and synthesis completed in {time.time() - start_time:.2f}s")
    
    return response

In [19]:
# Define test question parameters
target_prompt = ["What are Apple’s goals for carbon neutrality?",
"How does Unilever manage plastic waste?",
"What steps has Coca-Cola taken to reduce water usage?"]

for querys in target_prompt:

    # Trigger search directly against our permanent vector storage graph
    response_payload = query_persistent_sustainability_rag(
        query=querys,
        persist_dir="./storage_index"
    )

    print("\n Grounded strategic response text output:")
    print("-" * 60)
    print(response_payload.response)
    print("-" * 60)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 443.78it/s]


Search and synthesis completed in 7.68s

 Grounded strategic response text output:
------------------------------------------------------------
**Apple’s Carbon‑Neutrality Goals**

- **Achieve carbon neutrality across the entire carbon footprint (value chain) by 2030**【Apple_Environmental_Progress_Report_2024.md】.  
- **Reduce emissions by 75 % relative to the 2015 baseline** as part of the 2030 target【Apple_Environmental_Progress_Report_2024.md】.  
- **Prioritize emissions reductions first;** any remaining unavoidable emissions will be offset with high‑quality carbon‑removal credits (e.g., the Restore Fund)【Apple_Environmental_Progress_Report_2024.md】.  
- **Corporate emissions have already been carbon neutral since April 2020**, achieved through 100 % renewable electricity and energy‑efficiency measures【Apple_Environmental_Progress_Report_2024.md】.  

These goals are aligned with the Intergovernmental Panel on Climate Change’s recommendation for global carbon neutrality【Apple_Environ

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 458.93it/s]


Search and synthesis completed in 14.22s

 Grounded strategic response text output:
------------------------------------------------------------
**Unilever’s approach to managing plastic waste**

| Area | Actions | Source |
|------|---------|--------|
| **Reduction of virgin plastic** | • Switches to recycled plastic, designs lighter packaging, and develops alternative materials, formats and models. <br>• Introduces new packaging solutions such as laundry sheets and capsules in cardboard boxes. | (Source File: unilever-annual-report-and-accounts-2024-243-293.md, chunk 194) |
| **Reuse‑refill pilots** | • Conducted > 50 reuse‑refill pilots worldwide since 2021 to test and learn from different consumer‑shopping and product‑use approaches. | (Source File: unilever-annual-report-and-accounts-2024-243-293.md, chunk 194) |
| **Collaboration & industry initiatives** | • Participates in industry‑wide initiatives led by the World Economic Forum, Ellen MacArthur Foundation, and Consumer Goods Fo

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 425.28it/s]


Search and synthesis completed in 24.76s

 Grounded strategic response text output:
------------------------------------------------------------
**Coca‑Cola’s actions to reduce water usage**

| Action | Evidence |
|--------|----------|
| Adopt a **local, water‑security‑focused approach** – promoting water stewardship and improving water use efficiency in its own operations. | “Because water is local, we take a local approach to increase water security. We're focused on promoting water stewardship, increasing water use efficiency in our operations and treating and returning safe water to communities.” (Source File: 2024-coco-cola-environmental-update.md, chunk_index: 19) |
| **Treat and return safe water** to the communities where it operates. | Same passage as above (chunk_index: 19). |
| **Achieve a 10 % improvement in water‑efficiency** across all system operations in 2024 compared with the 2015 baseline. | “10% OF WATER EFFICIENCY IMPROVEMENT ACROSS ALL SYSTEM OPERATIONS IN 2024, CO

### Step 4 : Evaluations using LLM-as-a-Judge

In [8]:
def run_capstone_triple_evaluation(query: str, response_obj: Any, index: Any) -> Dict[str, Any]:
    """
    Evaluates the RAG pipeline output using an LLM Judge framework across 
    all 3 required criteria: Faithfulness, Relevance, and Conciseness.
    
    FIX: Now explicitly uses the active, disk-backed index context layers.
    """
    # 1. Extract raw strings from the LlamaIndex response wrapper object
    response_text = response_obj.response
    contexts_list = [node.node.get_content() for node in response_obj.source_nodes]
    
    # 2. Initialize the native LlamaIndex judges BOUND to your loaded index configurations
    # This guarantees the automated judges see the exact same data as your query loop!
    faithfulness_judge = FaithfulnessEvaluator(llm=Settings.llm)
    relevance_judge = RelevancyEvaluator(llm=Settings.llm)
    
    print("   -> Evaluating generated response quality via automated Groq LLM Judges...")
    
    faith_check = faithfulness_judge.evaluate(
        query=query, response=response_text, contexts=contexts_list
    )
    rel_check = relevance_judge.evaluate(
        query=query, response=response_text, contexts=contexts_list
    )
    
    # 3. Custom LLM Judge Prompt to calculate the missing "Conciseness" metric
    conciseness_prompt = PromptTemplate(
        "You are an expert AI system evaluation judge.\n"
        "Analyze the provided Answer to see if it fulfills the Conciseness criteria.\n"
        "Conciseness Definition: Is the answer brief, highly clear, well-summarized, "
        "and completely free of unnecessary repetition or filler text?\n\n"
        "EVALUATION MATERIALS:\n"
        "Question asked: {query_str}\n"
        "Generated Answer: {answer_str}\n\n"
        "CRITICAL INSTRUCTIONS:\n"
        "Return your evaluation as a clean JSON block matching EXACTLY these keys:\n"
        "- score: (An integer value between 1 and 5, where 5 is perfectly concise and 1 is highly repetitive or bloated)\n"
        "- reasoning: (A single sentence explaining your score)\n"
    )
    
    try:
        raw_llm_json = Settings.llm.complete(
            conciseness_prompt.format(query_str=query, answer_str=response_text)
        ).text
        clean_json = raw_llm_json.replace("```json", "").replace("```", "").strip()
        conciseness_data = json.loads(clean_json)
    except Exception as e:
        conciseness_data = {"score": "N/A", "reasoning": f"Parser skipped or failed: {e}"}

    try:
        score_val = int(conciseness_data.get("score", 0))
        conciseness_passed = score_val >= 3
    except (ValueError, TypeError):
        conciseness_passed = False

    # 4. Compile all three criteria into the requested scorecard layout
    score_card = {
        "1. Faithfulness (Anti-Hallucination)": {
            "Assessment": "PASSED" if faith_check.passing else "FAILED",
            "Score (1-5 Scale)": 5 if faith_check.passing else 1,
            "Judge Explanation": getattr(faith_check, "feedback", "Factual alignment verified by Groq.")
        },
        "2. Relevance (Contextual Accuracy)": {
            "Assessment": "PASSED" if rel_check.passing else "FAILED",
            "Score (1-5 Scale)": 5 if rel_check.passing else 1,
            "Judge Explanation": getattr(rel_check, "feedback", "Response directly answers user prompt intent.")
        },
        "3. Conciseness (Summary Efficiency)": {
            "Assessment": "PASSED" if conciseness_passed else "FAILED",
            "Score (1-5 Scale)": conciseness_data.get("score"),
            "Judge Explanation": conciseness_data.get("reasoning")
        }
    }
    
    return score_card

In [49]:
# =====================================================================
# AUTOMATED BATCH RUNNER LOOP (REPLACES THE MANUAL INPUT SELECTION)
# =====================================================================

# Define test question parameters
target_prompts: List[str] = [
    "What are Apple’s goals for carbon neutrality?",
    "How does Unilever manage plastic waste?",
    "What steps has Coca-Cola taken to reduce water usage?"
]

print(f"🚀 Initializing automated execution and evaluation for {len(target_prompts)} corporate queries...")

for idx, current_query in enumerate(target_prompts, 1):
    print(f"\n============================================================")
    print(f" PROCESSING QUERY {idx}/{len(target_prompts)}: '{current_query}'")
    print(f"============================================================")
    
    # 1. Trigger search directly against our permanent vector storage index 
    response_payload = query_persistent_sustainability_rag(
        query=current_query,
        index=sustainability_vector_index
    )
    
    # 2. Print out the raw pipeline response string
    print("\n Grounded strategic response text output:")
    print("-" * 60)
    print(response_payload.response)
    print("-" * 60)
    
    # 3. Automatically run the evaluation scorecard
    print("\n Running automated metric suite...")
    # FIXED: Added the 'index' mapping argument to keep the tracking unified
    evaluation_scorecard = run_capstone_triple_evaluation(
        query=current_query, 
        response_obj=response_payload,
        index=sustainability_vector_index
    )
    
    # 4. Display the resulting scorecard dashboard directly below the text logs
    print(f"\n Capstone Evaluation Scorecard for Query {idx}:")
    print(json.dumps(evaluation_scorecard, indent=4))

🚀 Initializing automated execution and evaluation for 3 corporate queries...

 PROCESSING QUERY 1/3: 'What are Apple’s goals for carbon neutrality?'


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 766.81it/s]


Search and synthesis completed in 4.55s

 Grounded strategic response text output:
------------------------------------------------------------
**Apple’s carbon‑neutrality goals**

| Goal | Description | Source |
|------|-------------|--------|
| **Carbon neutrality across the entire value chain by 2030** | Apple commits to balancing all remaining unavoidable emissions with high‑quality carbon‑removal credits to achieve net‑zero across its whole carbon footprint. | (Apple_Environmental_Progress_Report_2024.md, chunk_index: 35) |
| **75 % emissions reduction vs. 2015 baseline** | The company aims to cut its greenhouse‑gas emissions by three‑quarters relative to its 2015 footprint as a prerequisite to reaching net‑zero. | (Apple_Environmental_Progress_Report_2024.md, chunk_index: 160) |
| **Prioritization of emissions reductions over removal** | Reductions in facilities and supply‑chain emissions are the primary focus; carbon removal is used only for emissions that cannot yet be avoided.

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 729.28it/s]


Search and synthesis completed in 5.39s

 Grounded strategic response text output:
------------------------------------------------------------
**Unilever’s approach to managing plastic waste**

| Action | Description | Source |
|--------|-------------|--------|
| **Reduce virgin‑plastic use** | Switch to recycled plastic, design lighter packaging, and develop alternative materials and formats. | (Source File: unilever-annual-report-and-accounts-2024-243-293.md, Chunk: 194) |
| **Introduce plastic‑free or lower‑plastic solutions** | Launch laundry sheets and capsules packaged in cardboard boxes to eliminate or cut plastic. | (Source File: unilever-annual-report-and-accounts-2024-243-293.md, Chunk: 194) |
| **Reuse‑refill pilots** | Conducted > 50 pilots worldwide since 2021 to test reusable and refillable packaging models. | (Source File: unilever-annual-report-and-accounts-2024-243-293.md, Chunk: 194) |
| **Collaborate on industry initiatives** | Participate in programs led by the Wor

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 667.97it/s]


Search and synthesis completed in 4.77s

 Grounded strategic response text output:
------------------------------------------------------------
**Coca‑Cola’s actions to reduce water usage**

| Action | Evidence |
|--------|----------|
| Adopt a **local approach** to water security, tailoring initiatives to each community. | “Because water is local, we take a local approach to increase water security.” (Source File: 2024-coco-cola-environmental-update.md, chunk_index: 19) |
| **Promote water stewardship** and improve water‑use efficiency across its operations. | “We're focused on promoting water stewardship, increasing water use efficiency in our operations…” (Source File: 2024-coco-cola-environmental-update.md, chunk_index: 19) |
| **Treat and return safe water** to communities and the environment. | “…treating and returning safe water to communities.” (Source File: 2024-coco-cola-environmental-update.md, chunk_index: 19) |
| **Replenish water** by returning more water to nature and co

### Step 5 : FastAPI Route the requests to serve the user query in Streamlit

In [9]:

import threading
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Jupyter RAG Bridge API")

class QueryPayload(BaseModel):
    question: str

@app.get("/")
def home():
    return {"status": "connected", "message": "Jupyter RAG engine is live!"}

@app.post("/ask")
def ask_rag_engine(payload: QueryPayload):
    try:
        # Calls the updated function using disk-backed storage path
        response_output = query_persistent_sustainability_rag(
            query=payload.question,
            persist_dir="./storage_index"
        )
        
        sources_list = []
        for node in response_output.source_nodes:
            sources_list.append({
                "file_name": node.node.metadata.get("source_file", "Unknown"),
                "reranker_score": float(node.score) if node.score is not None else 0.0,
                "text": node.node.get_content()
            })
            
        try:
            scorecard = run_capstone_triple_evaluation(
                query=payload.question, 
                response_obj=response_output,
                index="./storage_index"

            )
        except Exception as eval_err:
            scorecard = {"error": f"Evaluation extraction skipped: {eval_err}"}
        
        return {
            "answer": response_output.response,
            "sources": sources_list,
            "scorecard": scorecard
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))



In [ ]:
# --- THREAD-SAFE SERVER RUNNER ENGINE ---
def run_server():
    # Force uvicorn to run on standard loop configurations inside its own thread sandbox
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()
    

# Check if a server thread is already running to prevent spinning up duplicate servers on double-clicks
if not any(t.name == "RAG_Server_Thread" for t in threading.enumerate()):
    print("Initializing isolated background thread sandbox for Uvicorn server...")
    server_thread = threading.Thread(target=run_server, name="RAG_Server_Thread", daemon=True)
    server_thread.start()
    print("Local Bridge API Server is now running silently in the background on http://127.0.0.1:8000")
else:
    print("Server thread is already active and listening for incoming Streamlit payloads on port 8000!")


Server thread is already active and listening for incoming Streamlit payloads on port 8000!


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 323.41it/s]


Search and synthesis completed in 17.45s
   -> Evaluating generated response quality via automated Groq LLM Judges...
INFO:     127.0.0.1:58688 - "POST /ask HTTP/1.1" 200 OK


In [ ]:
# ### Refined Code


# def run_capstone_triple_evaluation(query: str, response_obj: Any) -> Dict[str, Any]:
#     """
#     Evaluates the RAG pipeline output using an LLM Judge framework across 
#     all 3 required criteria: Faithfulness, Relevance, and Conciseness.
#     """
#     # 1. Extract raw strings from the LlamaIndex response wrapper object
#     response_text = response_obj.response
#     contexts_list = [node.node.get_content() for node in response_obj.source_nodes]
#     context_str = "\n\n".join(contexts_list)
    
#     # 2. Initialize the native LlamaIndex judges (Faithfulness & Relevance)
#     faithfulness_judge = FaithfulnessEvaluator()
#     relevance_judge = RelevancyEvaluator()
    
#     print(" Evaluating generated response quality via automated Groq LLM Judges...")
    
#     faith_check = faithfulness_judge.evaluate(
#         query=query, response=response_text, contexts=contexts_list
#     )
#     rel_check = relevance_judge.evaluate(
#         query=query, response=response_text, contexts=contexts_list
#     )
    
#     # 3. Custom LLM Judge Prompt to calculate the missing "Conciseness" metric
#     # Maps directly to standard LLM evaluation design methodologies
#     conciseness_prompt = PromptTemplate(
#         "You are an expert AI system evaluation judge.\n"
#         "Analyze the provided Answer to see if it fulfills the Conciseness criteria.\n"
#         "Conciseness Definition: Is the answer brief, highly clear, well-summarized, "
#         "and completely free of unnecessary repetition or filler text?\n\n"
#         "EVALUATION MATERIALS:\n"
#         "Question asked: {query_str}\n"
#         "Generated Answer: {answer_str}\n\n"
#         "CRITICAL INSTRUCTIONS:\n"
#         "Return your evaluation as a clean JSON block matching EXACTLY these keys:\n"
#         "- score: (An integer value between 1 and 5, where 5 is perfectly concise and 1 is highly repetitive or bloated)\n"
#         "- reasoning: (A single sentence explaining your score)\n"
#     )
    
#     # Trigger Groq to score conciseness using our structural guidelines
#     try:
#         raw_llm_json = Settings.llm.complete(
#             conciseness_prompt.format(query_str=query, answer_str=response_text)
#         ).text
#         # Clean any accidental string markdown code block tags if returned by the LLM
#         clean_json = raw_llm_json.replace("```json", "").replace("```", "").strip()
#         conciseness_data = json.loads(clean_json)
#     except Exception as e:
#         conciseness_data = {"score": "N/A", "reasoning": f"Parser skipped or failed: {e}"}

#     # 4. Compile all three criteria into the requested scorecard layout
#     score_card = {
#         "1. Faithfulness (Anti-Hallucination)": {
#             "Assessment": "PASSED" if faith_check.passing else "FAILED",
#             "Score (1-5 Scale)": 5 if faith_check.passing else 1,
#             "Judge Explanation": getattr(faith_check, "feedback", "Factual alignment verified by Groq.")
#         },
#         "2. Relevance (Contextual Accuracy)": {
#             "Assessment": "PASSED" if rel_check.passing else "FAILED",
#             "Score (1-5 Scale)": 5 if rel_check.passing else 1,
#             "Judge Explanation": getattr(rel_check, "feedback", "Response directly answers user prompt intent.")
#         },
#         "3. Conciseness (Summary Efficiency)": {
#             "Assessment": "PASSED" if conciseness_data.get("score", 0) >= 3 else "FAILED",
#             "Score (1-5 Scale)": conciseness_data.get("score"),
#             "Judge Explanation": conciseness_data.get("reasoning")
#         }
#     }
    
#     return score_card

In [ ]:
# # Hardcode the question string matching the query executed in Cell 8
# MANUAL_QUESTION_STR = input("Type your question : ")

# evaluation_scorecard = run_capstone_triple_evaluation(
#     query=MANUAL_QUESTION_STR, 
#     response_obj=response_payload
# )

# print("\n Capstone Project Qualitative Evaluation Scorecard Output:\n")
# print(json.dumps(evaluation_scorecard, indent=4))